# Lab 04: From LLM to Agent -- SOLUTION

**Goal:** See the same task handled by a plain LLM vs an "agent" with tools.

**What you'll learn:**
- A plain LLM can only guess at answers requiring live data
- An "agent" (LLM + tools) can fetch real data and give accurate answers
- Tools bridge the gap between "knowing" and "doing"

**Scenarios:** Inventory lookup, compound interest, currency exchange, and multi-tool investment analysis

In [ ]:
import math
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOllama(model="llama3.2:1b")

In [ ]:
# Tools
def inventory_tool(product: str) -> str:
    """Returns current stock level for a product from the warehouse database."""
    inventory = {
        "Laptop Pro": "In Stock — 142 units",
        "Wireless Mouse": "Low Stock — 8 units",
        "USB-C Hub": "Out of Stock — 0 units",
    }
    return inventory.get(product, "Product not found")


def finance_tool(expression: str) -> str:
    """Safely evaluate a financial/math expression."""
    try:
        result = eval(expression, {"__builtins__": {}, "math": math})
        return f"{result:.2f}"
    except Exception as e:
        return f"Error: {e}"


def exchange_rate_tool(pair: str) -> str:
    """Simulated forex API (in real life, this calls an exchange rate service)."""
    rates = {
        "INR-USD": "1 INR = 0.012 USD",
        "INR-EUR": "1 INR = 0.011 EUR",
        "USD-INR": "1 USD = 83.50 INR",
        "JPY-INR": "1 JPY = 0.56 INR",
    }
    return rates.get(pair, "Currency pair not available")

## Scenario 1: Stock Level of Laptop Pro

### Plain LLM

In [ ]:
print(f"LLM: {llm.invoke('What is the current stock level of Laptop Pro? Give specific numbers.').content}")

### Agent

In [ ]:
s = inventory_tool("Laptop Pro")
r = llm.invoke([
    SystemMessage(content="Use the data provided to answer accurately."),
    HumanMessage(content=f"Inventory data: {s}\nWhat is the current stock level of Laptop Pro?"),
])
print(f"Agent: {r.content} (tool: {s})")

## Scenario 2: Compound Interest

### Plain LLM

In [ ]:
print(f"LLM: {llm.invoke('Rs 10,000 at 8% compound interest for 5 years — exact final amount?').content}")
print(f"Correct: {10000 * (1 + 0.08) ** 5:.2f}")

### Agent

In [ ]:
calc = finance_tool("10000 * (1 + 0.08) ** 5")
r = llm.invoke([
    SystemMessage(content="Use the calculation result to answer."),
    HumanMessage(content=f"Compound interest: 10000 * (1.08)^5 = Rs {calc}\nHow much will Rs 10,000 become at 8% for 5 years?"),
])
print(f"Agent: {r.content} (tool: Rs {calc})")

## Scenario 3: Currency Exchange

### Plain LLM

In [ ]:
print(f"LLM: {llm.invoke('What is 5000 INR in USD? Give the exact amount.').content}")

### Agent

In [ ]:
rate = exchange_rate_tool("INR-USD")
r = llm.invoke([
    SystemMessage(content="Use the exchange rate provided to answer."),
    HumanMessage(content=f"Exchange rate: {rate}\nWhat is 5000 INR in USD?"),
])
print(f"Agent: {r.content} (tool: {rate})")

## TODO 1: Recipe Scaling Comparison

In [ ]:
def recipe_tool(dish: str) -> str:
    """Returns ingredients for a dish (serves 4)."""
    recipes = {
        "Paneer Butter Masala": "Paneer 250g, Butter 50g, Tomato Puree 200ml, Cream 100ml, Onion 2, Ginger-Garlic Paste 2 tbsp",
        "Dal Tadka": "Toor Dal 200g, Ghee 30g, Cumin Seeds 1 tsp, Onion 1, Tomato 2, Green Chili 2",
        "Vegetable Biryani": "Basmati Rice 300g, Mixed Vegetables 400g, Yogurt 100g, Biryani Masala 2 tbsp, Onion 3, Oil 60ml",
    }
    return recipes.get(dish, "Recipe not found")

### Plain LLM

In [ ]:
print(f"LLM: {llm.invoke('What ingredients do I need for Paneer Butter Masala for 8 people? List with quantities.').content}")

### Agent

In [ ]:
recipe = recipe_tool("Paneer Butter Masala")
r = llm.invoke([
    SystemMessage(content="Use the recipe data provided. The recipe is for 4 servings. Scale all quantities to the requested serving size."),
    HumanMessage(content=f"Recipe for Paneer Butter Masala (serves 4): {recipe}\n\nWhat ingredients do I need for Paneer Butter Masala for 8 people?"),
])
print(f"Agent: {r.content}")
print(f"(Tool: recipe_tool -> {recipe} [for 4], scaled to 8)")

## TODO 2: Trip Budget Multi-Tool

In [ ]:
# Step 1: Calculate total hotel cost in JPY
total_jpy = finance_tool("15000 * 5")
# Step 2: Get exchange rate
jpy_rate = exchange_rate_tool("JPY-INR")
# Step 3: Convert to INR (1 JPY = 0.56 INR)
total_inr = finance_tool(f"{total_jpy} * 0.56")

r = llm.invoke([
    SystemMessage(content="Use the data provided to give a complete answer."),
    HumanMessage(content=f"Hotel cost: 15,000 JPY/night x 5 nights = {total_jpy} JPY\nExchange rate: {jpy_rate}\nTotal in INR: Rs {total_inr}\n\nIf a hotel in Tokyo costs 15,000 JPY per night for 5 nights, how much is that in INR?"),
])
print(f"Agent: {r.content}")
print(f"(Tool 1: finance -> {total_jpy} JPY)")
print(f"(Tool 2: exchange rate -> {jpy_rate})")
print(f"(Tool 3: finance -> Rs {total_inr})")

## Key Takeaways

- **Plain LLM:** Smart but blind (no access to live data)
- **Agent (LLM + Tools):** Smart AND connected to the world
- Tools give the LLM real information to work with
- The LLM's job is to **REASON**; tools provide the **DATA**
- This is the foundation of all agentic AI!